# Ingest NutriAccess Food Desert

This notebook connects to the project S3 bucket, defines the project path, verifies the raw datasets exist, and loads the tabular datasets into pandas for exploration.

## **Data Sources:**
### 1. Food Access Research Atlas 

https://www.ers.usda.gov/data-products/food-access-research-atlas

Source: United States Department of Agriculture (USDA)
Identify communities with limited access to healthy food retailers using census-
tract-level indicators such as supermarket distance, vehicle access, and income
status

The dataset contains census-tract-level food access indicators

    - distance to the nearest supermarket
    - low-income populations
    - vehicle access
    - food desert indicators
    - ~70,000+ census tracts

Purpose in project: Identify communities with limited access to healthy food
retailers.

### 2. Census Demographic Data

https://www.census.gov/data.html

Source: U.S. Census Bureau
Analyze how income and demographics, such as income level, population density,
Poverty rates and household composition are related to food desert prevalence.

Useful tables

    - Income
    - population density
    - poverty rate

Download tables (filter for United States) 

    S1901: Income in the Past 12 Months
    S1701: Poverty Status
    DP05: Demographic and Housing Estimates

Purpose in the project: Analyze how income and demographics relate to food
desert prevalence.

### 3. Retail Food Environment Data

https://www.ers.usda.gov/data-products/food-environment-atlas

Source: USDA Food Environment Atlas
Evaluate the balance of healthy vs. unhealthy food retailers.

The dataset includes:

    - grocery store availability
    - fast food density
    - farmers markets
    - SNAP access
    - thousands of counties with many variables

Dataset coverage: thousands of U.S. counties with multiple food environments
indicators.
Purpose in project: Evaluate the balance of healthy vs. unhealthy food
retailers.

### 4. Health Outcomes Dataset

https://www.cdc.gov/places/tools/data-portal.html

Source: CDC PLACES dataset
Investigate potential relationships between food access and community health
outcomes, including obesity prevalence, diabetes rates, heart disease, and physical
inactivity.

local health estimates for U.S. communities

    - obesity prevalence
    - diabetes rates
    - heart disease
    - physical inactivity

Purpose in project: Investigate potential relationships between food access and
community health outcomes.

### 5. Grocery Store Location Data (optional enrichment data)

The GIS store location dataset is processed in a separate notebook and is not ingested here.

#### NOTE: Raw Datafiles for this project were downloaded on March 3rd, 2026, at 7:00 am 

# Clean and Auto-Ingest Data from an S3 bucket Dataset 

## Import Libraries 

In [2]:
import boto3
import pandas as pd 
import os
from dotenv import load_dotenv

## Define Project Configuration and Connect to S3 

In [3]:
load_dotenv()

AWS_REGION = os.getenv("AWS_REGION")
BUCKET = os.getenv("S3_BUCKET")
RAW_PREFIX = os.getenv("S3_PREFIX")
LOCAL_DATA_DIR = os.getenv("LOCAL_DATA_DIR")

if not BUCKET:
    raise ValueError("S3_BUCKET is not set in .env")

if not RAW_PREFIX:
    raise ValueError("S3_PREFIX is not set in .env")

s3 = boto3.client("s3", region_name=AWS_REGION)

## List every file that exists in the  raw data folder

*Note: Many files will appear; some interact with each other and remain part of the same dataset. For example, geospatial datasets contain many interacting files.*

In [4]:
# Use paginator in case there are many files
paginator = s3.get_paginator("list_objects_v2")

pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=RAW_PREFIX
)

files = []

for page in pages:
    for obj in page.get("Contents", []):
        files.append(obj["Key"])

print("Files found in rawData:")
for f in files:
    print(f)

Files found in rawData:
rawData/
rawData/ACSDP1Y2024.DP05-2026-03-13T140903.csv
rawData/ACSST1Y2024.S1701-2026-03-13T140807.csv
rawData/ACSST1Y2024.S1901-2026-03-13T140835.csv
rawData/Censue_Dataset/ACSDP5Y2023.DP05-Data.csv
rawData/Censue_Dataset/ACSST5Y2023.S1701-Data.csv
rawData/Censue_Dataset/ACSST5Y2023.S1901-Data.csv
rawData/FoodAccess/
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/ReadMe.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/VariableLookup.csv
rawData/FoodEnvironment/
rawData/FoodEnvironment/2025-food-environment-atlas-data/ReadMeFile2025.txt
rawData/FoodEnvironment/2025-food-environment-atlas-data/StateAndCountyData.csv
rawData/FoodEnvironment/2025-food-environment-atlas-data/VariableList.csv
rawData/PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260313.csv
rawData/geofabrik_NorCal/
rawData/geofabrik_NorCal/norcal-260312-free.shp/README
rawDa

## Automatically Load useful CSV datasets

*Note: even .csv files that exist in subfolders will load* 

In [5]:
datasets = {}

skip_terms = ["readme", "variablelookup", "variablelist"]

for key in files:
    lower_key = key.lower()

    if key.endswith(".csv") and not any(term in lower_key for term in skip_terms):
        path = f"s3://{BUCKET}/{key}"
        print(f"Loading {path}")

        # Create a cleaner dataset name
        name = key.split("/")[-1].replace(".csv", "")
        name = name.replace(" ", "_").replace(",", "").replace("-", "_").replace(".", "_")


        datasets[name] = pd.read_csv(path, low_memory=False)

print("\nDatasets loaded:")
print(list(datasets.keys()))

Loading s3://nutriaccess-data/rawData/ACSDP1Y2024.DP05-2026-03-13T140903.csv
Loading s3://nutriaccess-data/rawData/ACSST1Y2024.S1701-2026-03-13T140807.csv
Loading s3://nutriaccess-data/rawData/ACSST1Y2024.S1901-2026-03-13T140835.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSDP5Y2023.DP05-Data.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSST5Y2023.S1701-Data.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSST5Y2023.S1901-Data.csv
Loading s3://nutriaccess-data/rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv
Loading s3://nutriaccess-data/rawData/FoodEnvironment/2025-food-environment-atlas-data/StateAndCountyData.csv
Loading s3://nutriaccess-data/rawData/PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260313.csv

Datasets loaded:
['ACSDP1Y2024_DP05_2026_03_13T140903', 'ACSST1Y2024_S1701_2026_03_13T140807', 'ACSST1Y2024_S1901_2026_03_13T140835', 'ACSDP5Y2023_DP05_Data', 'ACSST5Y2023_S1701_Data', 

## Release Resources

In [6]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [1]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>